<br>

# POST por Código do IBGE

<!--<badge>--><a href="https://colab.research.google.com/github/open-geodata/pyFBDS/blob/main/docs/scripts/post/03_post_multiple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><!--</badge>-->

Este notebook demonstra o fluxo para obter dados espaciais da FBDS a partir de um código de município do IBGE. Primeiro, o código é associado ao nome e à UF do município; em seguida, o pacote baixa o arquivo `.tar` pela abordagem HTTP `POST`, carrega um _layer_ como `GeoDataFrame` e o exibe em um mapa interativo.

A célula de instalação abaixo é opcional quando `pyFBDS` já estiver instalado. Para executar no Google Colab, use o badge acima e ajuste o caminho do projeto na seção **Pastas**.

<br>

Michel Metran\
Data: 21.09.2026\
Atualizado em: 13.11.2026


In [ ]:
# Descomente, para instalar
#!pip3 install pyFBDS

In [ ]:
from pathlib import Path

import geopandas as gpd
import xyzservices.providers as xyz

from pyFBDS.logger import FBDSLogger
from pyFBDS.lxml import FBDS as FBDS_xml
from pyFBDS.post import FBDS, IBGE

<br>

---

## Pastas

Define os diretórios de trabalho. `data/log` recebe os registros do logger; `data/output` é usado para o cache HTTP e para salvar os arquivos `.tar` baixados. As pastas são criadas automaticamente caso não existam.

O caminho inicial considera a estrutura local do repositório. No Colab, ajuste `project_path` para a pasta onde o notebook está sendo executado.


In [ ]:
project_path = Path(".").absolute().parents[2]

# No Colab, descomente essa
# project_path = Path('.').absolute()

# Diretório de saída
data_path = project_path / "data"

logs_path = data_path / "log"
logs_path.mkdir(parents=True, exist_ok=True)

output_path = data_path / "output"
output_path.mkdir(parents=True, exist_ok=True)
output_path

<br>

---

## FBDS

A primeira classe `FBDS_xml` é usada pela classe `IBGE` para consultar o portal e localizar os municípios disponíveis. A `IBGE` carrega uma planilha publicada pela FBDS com os códigos do IBGE, nomes dos municípios e UFs; `ibge.data.head()` mostra uma amostra dessa tabela.

### Busca pelo código do IBGE

`ibge.search(id_ibge=...)` procura o código na tabela, normaliza o nome do município e o compara com os nomes das pastas no portal. O retorno é uma tupla `(municipio, uf)`, usada nas etapas seguintes. Se não houver correspondência única, a busca pode apresentar sugestões ou informar que é necessário resolver a ambiguidade.

No exemplo, o código atualmente ativo é `3545803`, correspondente a Santa Bárbara d'Oeste (SP). A célula contém outras opções comentadas; para escolher outra, deixe ativa apenas a atribuição desejada.


In [ ]:
# Instancia Logger
# Inicializa o logger uma única vez antes do loop
logger = FBDSLogger(
    log_dir=logs_path,
    new_session=True,
    console=True,
)

# Instancia FBDS do módulo LXML
fbds = FBDS_xml(
    temp_path=output_path,
    output_path=output_path,
    logger=logger,
)

In [ ]:
ibge = IBGE(
    temp_path=output_path,
    fbds=fbds,
)

# Lê tabela
ibge.data.head()

Cria uma lista de municípios...


In [ ]:
list_municipios = []
list_municipios.append(
    {
        "id_municipio": 1506559,
        "municipio": "Santa Luzia do Pará",  # Opcional
    }
)
list_municipios.append(
    {
        "id_municipio": 1508308,
        "municipio": "Viseu",  # Opcional
    }
)

# Results
list_municipios

In [ ]:
list_gdfs = []
# Para cada município
for mun in list_municipios:
    id_ibge = mun["id_municipio"]
    nome_municipio = mun["municipio"]
    print(id_ibge, nome_municipio)

    # Instancia Logger
    # Inicializa o logger uma única vez antes do loop
    logger = FBDSLogger(
        log_dir=output_path,
        new_session=True,
        console=True,
    )

    # Instancia FBDS do módulo LXML
    fbds = FBDS_xml(
        temp_path=output_path,
        output_path=output_path,
        logger=logger,
    )

    ibge = IBGE(
        temp_path=output_path,
        fbds=fbds,
    )

    # Lê tabela
    ibge.data.head()

    municipio, uf = ibge.search(id_ibge=id_ibge)

    # Instancia FBDS
    fbds = FBDS(
        temp_path=output_path,
        output_path=output_path,
        logger=logger,
    )

    filepath = fbds.download(
        uf=uf,
        municipality=municipio,
        max_download_seconds=60 * 5,
    )

    list_shps = fbds.clean_list(filepath=filepath)

    # Parâmetro
    # layer = "MASSAS_DAGUA"
    # layer = "APP_USO"
    layer = "APP"

    if layer in list_shps:
        # Cria geodataframe
        gdf = fbds.read_data(
            uf=uf,
            municipality=municipio,
            layer=layer,
        )

        # gdf = gdf.dissolve()

        gdf = gdf.drop(
            labels=[
                "HIDRO",
                "APP_M",
                "AREA_HA",
                "GEOCODIGO",
                "MUNICIPIO",
                "UF",
                "CD_UF",
            ],
            errors="ignore",
            axis="columns",
        )
        list_gdfs.append(gdf)

    else:
        raise Exception(list_shps)

In [ ]:
# Junta
gdf_final = gpd.pd.concat(
    objs=list_gdfs,
    ignore_index=True,
)

# Garantir que o resultado continue sendo um GeoDataFrame válido
gdf_fbds = gpd.GeoDataFrame(
    data=gdf_final,
    crs=list_gdfs[0].crs,
)

gdf_fbds = gdf_fbds.dissolve()

# Results
gdf_fbds.info()
gdf_fbds.head()

Visualizamos o resultado.


In [ ]:
gdf_fbds.explore(
    tiles=xyz.Esri.WorldImagery,
)

E salvamos!


In [ ]:
gdf_fbds.to_file(
    filename=output_path / "fbds_app.shp",
    driver="ESRI Shapefile",
    encoding="UTF-8",
)

Se estiver no Google Colab, é possível copiar o arquivo pro GooGle Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

output_path = Path("/content/drive/MyDrive/FBDS")
output_path.mkdir(parents=True, exist_ok=True)

gdf_fbds.to_file(
    filename=output_path / "fbds_app.shp",
    driver="ESRI Shapefile",
    encoding="UTF-8",
)